In [ ]:
!pip install pyagrum

# Imports & global config

In [ ]:
import numpy as np
import pandas as pd
import pyagrum as gum
import pyagrum.lib.notebook as gnb

from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

STATE = ["Low", "Medium", "High"]
EPS = 1e-12


In [ ]:
import numpy as np
import pandas as pd
import pyagrum as gum
import pyagrum.lib.notebook as gnb

from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

STATE = ["Low", "Medium", "High"]
EPS = 1e-12

# ============================================================
# 0) (Optional) load strong rules from CSV
# ============================================================
# strong_task = pd.read_csv("strong_and_rules_task.csv")
# strong_rest = pd.read_csv("strong_and_rules_rest.csv")

# ============================================================
# 1) Parse rule -> conditions [(feat, state_idx)]
# ============================================================
def parse_rule_text(rule_text):
    """
    Expect: 'IF A is Low AND B is High THEN CL=Task'
    Return: [(A,0), (B,2)]
    """
    s = rule_text.strip()
    if "IF" in s:
        s = s.split("IF", 1)[1].strip()
    if "THEN" in s:
        s = s.split("THEN", 1)[0].strip()

    parts = [p.strip() for p in s.split("AND")]
    conds = []
    for p in parts:
        if " is " not in p:
            continue
        feat, st = [x.strip() for x in p.split(" is ", 1)]
        if st not in STATE:
            raise ValueError(f"Unknown state '{st}' in rule: {rule_text}")
        conds.append((feat, STATE.index(st)))
    return conds

def strong_features_from_rules(*dfs):
    feats = set()
    for df in dfs:
        for r in df["Rule"].tolist():
            for f,_ in parse_rule_text(r):
                feats.add(f)
    return sorted(list(feats))

STRONG_FEATURES = strong_features_from_rules(strong_task, strong_rest)
print("Strong features used as evidence:", STRONG_FEATURES)

# ============================================================
# 2) X_fuzzy -> beliefs per feature (normalize 3 states)
# ============================================================
def fuzzy_to_beliefs(X_fuzzy, columns, n_mf=3, eps=1e-6):
    beliefs = {}
    for i, feat in enumerate(columns):
        mu = X_fuzzy[:, i*n_mf:(i+1)*n_mf].copy()         # (N,3)
        mu = mu / (mu.sum(axis=1, keepdims=True) + eps)   # normalize
        # tránh 0 hoàn toàn
        mu = np.clip(mu, eps, 1.0)
        mu = mu / mu.sum(axis=1, keepdims=True)
        beliefs[feat] = mu
    return beliefs


# ============================================================
# 3) Group by sensor: HR -> {HR_mean, HR_std, HR_slope}, etc.
#    Build sensor beliefs from its 3 children (geometric mean across same state)
# ============================================================
def group_features_by_sensor(columns):
    groups = {}
    for feat in columns:
        sensor = feat.split("_")[0]  # TEMP_mean -> TEMP
        groups.setdefault(sensor, []).append(feat)
    # keep only sensors that have 3 features (mean/std/slope)
    groups = {k: sorted(v) for k,v in groups.items()}
    return groups

SENSOR_GROUPS = group_features_by_sensor(columns)
print("Sensor groups:", SENSOR_GROUPS)

def build_sensor_beliefs(feature_beliefs, sensor_groups, eps=1e-9):
    """
    feature_beliefs[feat]: (N,3)
    sensor_beliefs[sensor]: (N,3)
    Combine 3 child features using geometric mean per state:
      mu_sensor[s] ∝ (mu_f1[s] * mu_f2[s] * mu_f3[s])^(1/3)
    """
    sensor_beliefs = {}
    for sensor, feats in sensor_groups.items():
        # feats like [HR_mean, HR_slope, HR_std] (order doesn't matter)
        M = np.stack([feature_beliefs[f] for f in feats], axis=0)  # (K,N,3)
        # geometric mean across K features
        gm = np.exp(np.mean(np.log(M + eps), axis=0))              # (N,3)
        gm = gm / (gm.sum(axis=1, keepdims=True) + eps)
        sensor_beliefs[sensor] = gm
    return sensor_beliefs

# ============================================================
# 4) TAN edges between SENSORS using conditional MI I(Si;Sj|CL)
# ============================================================
def mi_from_joint(joint):
    pa = joint.sum(axis=1, keepdims=True)
    pb = joint.sum(axis=0, keepdims=True)
    ratio = (joint + EPS) / (pa @ pb + EPS)
    return float((joint * np.log(ratio + EPS)).sum())

def cond_mi_soft(belA, belB, y01, alpha=1.0):
    """
    belA, belB: (N,3) soft beliefs for two sensors
    y01: (N,) in {0,1}
    """
    y01 = np.asarray(y01).astype(int)
    total = 0.0
    for cls in [0,1]:
        mask = (y01 == cls)
        A = belA[mask]  # (Nc,3)
        B = belB[mask]  # (Nc,3)
        # expected joint counts
        joint_cnt = (A.T @ B) + alpha     # (3,3)
        joint = joint_cnt / (joint_cnt.sum() + EPS)
        Ic = mi_from_joint(joint)
        total += float(np.mean(y01==cls) * Ic)
    return float(total)

class DSU:
    def __init__(self, items):
        self.parent = {x:x for x in items}
        self.rank = {x:0 for x in items}
    def find(self, x):
        while self.parent[x]!=x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra==rb: return False
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra]==self.rank[rb]:
            self.rank[ra]+=1
        return True

def build_sensor_tan(sensor_beliefs, y01, alpha=1.0):
    sensors = list(sensor_beliefs.keys())
    edges = []
    for i in range(len(sensors)):
        for j in range(i+1, len(sensors)):
            a, b = sensors[i], sensors[j]
            w = cond_mi_soft(sensor_beliefs[a], sensor_beliefs[b], y01, alpha=alpha)
            edges.append((a,b,w))
    edges.sort(key=lambda x: -x[2])

    dsu = DSU(sensors)
    mst = []
    for a,b,w in edges:
        if dsu.union(a,b):
            mst.append((a,b,w))
        if len(mst)==len(sensors)-1:
            break

    # choose root by max incident weight sum
    adj = {s:[] for s in sensors}
    for a,b,w in mst:
        adj[a].append((b,w))
        adj[b].append((a,w))
    score = {s: sum(w for _,w in adj[s]) for s in sensors}
    root = max(sensors, key=lambda s: score[s])

    # direct tree by BFS
    parent = {root: None}
    q = [root]
    order = [root]
    while q:
        u = q.pop(0)
        for v,w in adj[u]:
            if v not in parent:
                parent[v] = u
                q.append(v)
                order.append(v)

    directed = []
    for v in order:
        if parent[v] is not None:
            directed.append((parent[v], v))
    return root, directed, mst

# ============================================================
# 5) Learn CPTs with soft counts
#    - P(Sensor | CL) or P(Sensor | CL, ParentSensor)
#    - P(Feature | Sensor)
# ============================================================
def learn_p_sensor_given_cl(sensor_beliefs, y01, alpha=1.0):
    y01 = np.asarray(y01).astype(int)
    cpt = {}
    for sensor, mu in sensor_beliefs.items():
        cpt[sensor] = {}
        for cls in [0,1]:
            mask = (y01 == cls)
            cnt = mu[mask].sum(axis=0) + alpha
            cpt[sensor][cls] = cnt / (cnt.sum() + EPS)
    return cpt

def learn_p_childsensor_given_cl_parent(sensor_beliefs, y01, parent, child, alpha=1.0):
    """
    return probs[cl, parent_state, child_state] shape (2,3,3)
    """
    y01 = np.asarray(y01).astype(int)
    P = sensor_beliefs[parent]  # (N,3)
    C = sensor_beliefs[child]   # (N,3)
    probs = np.zeros((2,3,3), dtype=float)

    for cl in [0,1]:
        mask = (y01==cl)
        PP = P[mask]
        CC = C[mask]
        for ps in range(3):
            w = PP[:, ps]  # (Nc,)
            cnt = (w[:, None] * CC).sum(axis=0) + alpha  # (3,)
            probs[cl, ps] = cnt / (cnt.sum() + EPS)
    return probs

def learn_p_feature_given_sensor(feature_beliefs, sensor_beliefs, sensor_groups, alpha=1.0):
    """
    return dict feature -> probs[sensor_state, feature_state] shape (3,3)
    """
    out = {}
    for sensor, feats in sensor_groups.items():
        muS = sensor_beliefs[sensor]  # (N,3)
        for feat in feats:
            muF = feature_beliefs[feat]  # (N,3)
            table = np.zeros((3,3), dtype=float)
            for ss in range(3):
                w = muS[:, ss]  # (N,)
                cnt = (w[:, None] * muF).sum(axis=0) + alpha
                table[ss] = cnt / (cnt.sum() + EPS)
            out[feat] = table
    return out

# ============================================================
# 6) Build BN structure: CL -> sensors; sensor TAN edges; sensor -> its features
# ============================================================
def build_bn_from_soft(
    columns,
    sensor_groups,
    tan_root, tan_edges,
    priors,
    cpt_sensor_root,          # P(sensor|CL) for all sensors
    cpt_sensor_child_dict,    # (parent,child) -> probs[cl,ps,cs]
    cpt_feat_given_sensor     # feature -> table[ss,fs]
):
    bn = gum.BayesNet("FuzzyBN_Sensor_TAN")

    # CL
    vcl = gum.LabelizedVariable("CL", "CognitiveLoad", 2)
    vcl.changeLabel(0, "Rest")
    vcl.changeLabel(1, "Task")
    bn.add(vcl)
    bn.cpt("CL").fillWith([float(priors[0]), float(priors[1])])

    # Sensors
    for sensor in sensor_groups.keys():
        v = gum.LabelizedVariable(sensor, sensor, 3)
        v.changeLabel(0, "Low"); v.changeLabel(1, "Medium"); v.changeLabel(2, "High")
        bn.add(v)
        bn.addArc("CL", sensor)

    # TAN arcs among sensors
    for src, dst in tan_edges:
        bn.addArc(src, dst)

    # Sensor CPTs
    for sensor in sensor_groups.keys():
        parents = [bn.variable(p).name() for p in bn.parents(sensor)]
        sensor_parents = [p for p in parents if p != "CL"]

        if len(sensor_parents)==0:
            # P(sensor_state | CL)
            bn.cpt(sensor)[{"CL":0}] = list(np.array(cpt_sensor_root[sensor][0], float))
            bn.cpt(sensor)[{"CL":1}] = list(np.array(cpt_sensor_root[sensor][1], float))
        else:
            # exactly 1 TAN parent
            p = sensor_parents[0]
            probs = cpt_sensor_child_dict[(p, sensor)]  # (2,3,3)
            for cl in [0,1]:
                for ps in [0,1,2]:
                    bn.cpt(sensor)[{"CL":cl, p:ps}] = list(probs[cl, ps])

    # Feature nodes (12)
    for feat in columns:
        v = gum.LabelizedVariable(feat, feat, 3)
        v.changeLabel(0, "Low"); v.changeLabel(1, "Medium"); v.changeLabel(2, "High")
        bn.add(v)

        sensor = feat.split("_")[0]
        bn.addArc(sensor, feat)

        # CPT P(feat | sensor)
        table = cpt_feat_given_sensor[feat]  # (3,3)
        for ss in [0,1,2]:
            bn.cpt(feat)[{sensor:ss}] = list(np.array(table[ss], float))

    return bn

# ============================================================
# 7) Inference: compute P(Task) with soft evidence on selected features
# ============================================================
def p_task_from_bn(bn, feature_beliefs, columns, use_features, engine="LazyPropagation"):
    """
    return p_task: (N,)
    """
    N = feature_beliefs[columns[0]].shape[0]

    ie = gum.LazyPropagation(bn) if engine == "LazyPropagation" else gum.ShaferShenoyInference(bn)

    p_task = np.zeros(N, dtype=float)

    for i in range(N):
        ie.eraseAllEvidence()

        # soft evidence (likelihoods)
        for feat in use_features:
            ie.addEvidence(feat, list(feature_beliefs[feat][i].astype(float)))

        ie.makeInference()

        post = ie.posterior("CL")  # gum.Potential

        # CÁCH 1 (ổn định nhất): lấy theo state index
        p_task[i] = float(post[{"CL": 1}])   # 1 = Task
        # p_rest  = float(post[{"CL": 0}])   # 0 = Rest

        # (Optional sanity check)
        # if abs((p_rest + p_task[i]) - 1.0) > 1e-6:
        #     print("Warning: posterior not normalized", p_rest, p_task[i])

    return p_task


# ============================================================
# 8) Train BN on (X_fuzzy,y) => returns bn + (p_task on given set)
# ============================================================
def fit_bn_and_predict(Xtr_fuzzy_fold, ytr_fold, Xva_fuzzy_fold, columns,
                       sensor_groups, evidence_features, alpha=1.0):
    # beliefs
    feat_bel_tr = fuzzy_to_beliefs(Xtr_fuzzy_fold, columns)
    feat_bel_va = fuzzy_to_beliefs(Xva_fuzzy_fold, columns)

    # sensor beliefs
    sen_bel_tr = build_sensor_beliefs(feat_bel_tr, sensor_groups)

    # TAN on sensors
    root, tan_edges, _ = build_sensor_tan(sen_bel_tr, ytr_fold, alpha=alpha)

    # priors
    priors = {0: float(np.mean(np.asarray(ytr_fold)==0)),
              1: float(np.mean(np.asarray(ytr_fold)==1))}

    # CPTs
    cpt_sensor_root = learn_p_sensor_given_cl(sen_bel_tr, ytr_fold, alpha=alpha)

    cpt_sensor_child_dict = {}
    for (p, c) in tan_edges:
        probs = learn_p_childsensor_given_cl_parent(sen_bel_tr, ytr_fold, p, c, alpha=alpha)
        cpt_sensor_child_dict[(p,c)] = probs

    cpt_feat_given_sensor = learn_p_feature_given_sensor(feat_bel_tr, sen_bel_tr, sensor_groups, alpha=alpha)

    # build BN
    bn = build_bn_from_soft(columns, sensor_groups, root, tan_edges, priors,
                            cpt_sensor_root, cpt_sensor_child_dict, cpt_feat_given_sensor)

    # inference on val
    p_task = p_task_from_bn(bn, feat_bel_va, columns, evidence_features)
    return bn, p_task

# ============================================================
# 9) GROUPKFOLD ON TRAIN
# ============================================================
def eval_groupkfold_bn(Xtr_fuzzy, y_train, user_train, columns,
                       sensor_groups, evidence_features,
                       n_splits=5, alpha=1.0):
    gkf = GroupKFold(n_splits=n_splits)
    rows = []

    y_train = np.asarray(y_train).astype(int)
    groups = np.asarray(user_train)

    for fold, (tr, va) in enumerate(gkf.split(Xtr_fuzzy, y_train, groups), start=1):
        _, p_task = fit_bn_and_predict(
            Xtr_fuzzy[tr], y_train[tr],
            Xtr_fuzzy[va], columns,
            sensor_groups, evidence_features,
            alpha=alpha
        )

        y_true = y_train[va]
        y_pred = (p_task >= 0.5).astype(int)

        acc = accuracy_score(y_true, y_pred)
        f1  = f1_score(y_true, y_pred)
        auc = roc_auc_score(y_true, p_task)

        # Brier score
        brier = float(np.mean((p_task - y_true)**2))

        rows.append({"fold": fold, "acc": acc, "f1": f1, "auc": auc, "brier": brier})
        print(f"[Fold {fold}] acc={acc:.3f} f1={f1:.3f} auc={auc:.3f} brier={brier:.3f}")

    df = pd.DataFrame(rows)
    print("\nCV mean:")
    display(df.mean(numeric_only=True))
    return df

# Choose evidence set:
# - chỉ dùng strong features => ít nhiễu, “đúng ý strong rules”
# - hoặc dùng all features => baseline đầy đủ
USE_ONLY_STRONG = True
EVIDENCE_FEATURES = STRONG_FEATURES if USE_ONLY_STRONG else columns
print("Evidence features:", EVIDENCE_FEATURES)

cv_df = eval_groupkfold_bn(
    Xtr_fuzzy_n, y_train, user_train, columns,
    SENSOR_GROUPS, EVIDENCE_FEATURES,
    n_splits=5, alpha=1.0
)

# ============================================================
# 10) FINAL: Train on ALL TRAIN -> Evaluate on HELD-OUT TEST
# ============================================================
bn_final, p_task_test = fit_bn_and_predict(
    Xtr_fuzzy_n, np.asarray(y_train).astype(int),
    Xte_fuzzy_n, columns,
    SENSOR_GROUPS, EVIDENCE_FEATURES,
    alpha=1.0
)

y_test = np.asarray(y_test).astype(int)
y_pred_test = (p_task_test >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred_test)
f1  = f1_score(y_test, y_pred_test)
auc = roc_auc_score(y_test, p_task_test)
brier = float(np.mean((p_task_test - y_test)**2))

print("\nFINAL TEST:")
print("Accuracy:", acc)
print("F1:", f1)
print("AUC:", auc)
print("Brier:", brier)
print("Confusion matrix:\n", pd.crosstab(y_test, y_pred_test, rownames=["True"], colnames=["Pred"]))

# ============================================================
# 11) Visualize BN like your example
# ============================================================
gnb.showBN(bn_final, size="10")
gnb.showInference(bn_final, evs={"CL":"Task"}, size="10")
gnb.showInference(bn_final, evs={"CL":"Rest"}, size="10")

# Optional: show inference with soft evidence of 1 test sample
# (pyAgrum notebook widget sometimes doesn't accept soft evidence in gnb.showInference,
#  so we'll just print posterior computed by engine)
def debug_one_sample_soft(bn, X_fuzzy, columns, evidence_features, idx=0):

    ie = gum.LazyPropagation(bn_final)
    ie.eraseAllEvidence()
    
    # giả lập soft evidence của sample 0
    bel = fuzzy_to_beliefs(X_fuzzy, columns)
    for feat in EVIDENCE_FEATURES:
        ie.addEvidence(feat, list(bel[feat][0].astype(float)))
    
    ie.makeInference()
    post = ie.posterior("CL")
    print("posterior type:", type(post))
    print("P(Rest):", float(post[{"CL":0}]))
    print("P(Task):", float(post[{"CL":1}]))


debug_one_sample_soft(bn_final, Xte_fuzzy_n, columns, EVIDENCE_FEATURES, idx=0)
